In [1]:
# Preparing input variable for clustering precipitation zones based on climatology of predictor -> predictand relationship

# V1 - 2026-10-26 - for final run - JPK

# Workflow
# stack -> aggregate -> reproject -> save
#FIX:: variable consideration for clustering requires revision

In [ ]:
#libraries
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rioxarray as rxr
import xarray as xr
import matplotlib.pyplot as plt
from rasterio.enums import Resampling
import dask
import sys

#paths
out_path = r"E:\ClimateData_Precipitation\Data\Processed\Spatial_clustering\Climatology_files"
os.makedirs(out_path, exist_ok=True)

sample_raster = rxr.open_rasterio(r"E:\ClimateData_Precipitation\Data\Datasets\Sample_raster\DEM_1km_elevation_max.tif")

#paths to datasets - chirps,  gldas, dem, fetch
base_dataset_path = r'E:\ClimateData_Precipitation\Data\Datasets' 
chirps_files_path = os.path.join(base_dataset_path, "CHIRPS")
gldas_files_path = os.path.join(base_dataset_path, "GLDAS")

fetch_rasters_path = r"E:\ClimateData_Precipitation\Data\Processed\Fetch"
fetch_rasters_temp_path = r"E:\ClimateData_Precipitation\Data\Processed\Fetch\Preprocessing"
dem_rasters_path = os.path.join(base_dataset_path,"DEM_1km")

# Helper - clip and resample raster to ROI at 1km resolution
def clip_and_resample_raster(input_raster):
    resampled_raster = input_raster.rio.reproject_match(sample_raster, resampling= Resampling.bilinear)
    return resampled_raster

In [3]:
# Create precipitation climatology from the finest resolution dataset -- CHIRPS

#tmax
chirps_files = sorted(glob.glob(os.path.join(chirps_files_path, "CHIRPS_prec_*.tif")))
chirps_dates = [pd.to_datetime(f[-12:-4], format='%Y%m%d') for f in chirps_files]

chirps_files_rxr = xr.concat([rxr.open_rasterio(f, chunks={'x': 1024, 'y': 1024}, masked = True) for f in chirps_files], 
               dim=pd.Index(chirps_dates, name='time'), join = 'exact')

prec_climatology = chirps_files_rxr.resample(time='1YE').sum().sum(dim='time')
prec_clim = clip_and_resample_raster(prec_climatology)
prec_clim.rio.to_raster(os.path.join(out_path, "prec_climatology.tif"))

del chirps_files, chirps_dates, chirps_files_rxr, prec_climatology, prec_clim  

In [4]:
# Resample fetch rasters

fetch_rasters = ["fetch_dir_500m_resampled.tif", "fetch_dist_500m_resampled.tif", 
                 "fetch_dir_1000m_resampled.tif", "fetch_dist_1000m_resampled.tif"]

for fetch_raster in fetch_rasters:
    read_raster = rxr.open_rasterio(os.path.join(fetch_rasters_path, fetch_raster))
    resampled_raster = clip_and_resample_raster(read_raster)
    resampled_raster.rio.to_raster(os.path.join(out_path, fetch_raster))
    del read_raster, resampled_raster

#prepare additional (temporary) fetch variables

fetch_temp_rasters = ["local_relief.tif", "dem_smoothed.tif"]

for fetch_temp_raster in fetch_temp_rasters:
    read_raster = rxr.open_rasterio(os.path.join(fetch_rasters_temp_path, fetch_temp_raster))
    resampled_raster = clip_and_resample_raster(read_raster)
    resampled_raster.rio.to_raster(os.path.join(out_path, fetch_temp_raster))
    del read_raster, resampled_raster

In [5]:
# Create GLDAS secondary variables climatology
gldas_variables = ['air_temp', 'lwdown', 'specific_humidity', 'surface_pressure', 'swdown', 'wind_speed']

for variable in gldas_variables:
    var_path = os.path.join(gldas_files_path, variable)
    var_files = sorted(glob.glob(os.path.join(var_path, "*.tif")))
    var_dates = [pd.to_datetime(f[-12:-4], format='%Y%m%d') for f in var_files]

    var_files_rxr = xr.concat([rxr.open_rasterio(f, chunks={'x': 1024, 'y': 1024}) for f in var_files], 
               dim=pd.Index(var_dates, name='time'), join = 'exact')
    var_climatology = var_files_rxr.resample(time='1YE').mean().mean(dim='time')
    var_clim = clip_and_resample_raster(var_climatology)
    var_clim.rio.to_raster(os.path.join(out_path, "GLDAS_" + variable + "_climatology.tif"))
    
    del var_files_rxr, var_climatology, var_clim

In [6]:
# Prepare DEM rasters

dem_rasters = ['DEM_1km_aspect_mean.tif', 'DEM_1km_elevation_max.tif', 
               'DEM_1km_elevation_mean.tif', 'DEM_1km_elevation_min.tif',
               'DEM_1km_elevation_std.tif', 'DEM_1km_slope_mean.tif']

for dem_raster in dem_rasters:
    read_raster = rxr.open_rasterio(os.path.join(dem_rasters_path, dem_raster))
    resampled_raster = clip_and_resample_raster(read_raster)
    resampled_raster.rio.to_raster(os.path.join(out_path, dem_raster))
    del read_raster, resampled_raster

In [8]:
# Prepare MODIS variables
from tqdm.notebook import tqdm 

#modis_variables = ['lst_n', 'lst_d', 'evi', 'lst_diff']
modis_variables = ['lst_diff']

for variable in tqdm(modis_variables, desc="Variables"):
    var_path = os.path.join(base_dataset_path, "MODIS_2_" + variable)
    var_files = sorted(glob.glob(os.path.join(var_path, "*.tif")))
    var_dates = [pd.to_datetime(f[-12:-4], format='%Y%m%d') for f in var_files]
    
    df = pd.DataFrame({'file': var_files, 'date': var_dates})
    df['year'] = df['date'].dt.year

    yearly_means = []
    for year, group in tqdm(list(df.groupby('year')), desc=f"Years ({variable})", leave=False):
        resampled_arrays = []
        for f, date in zip(group['file'], group['date']):
            arr = rxr.open_rasterio(f, chunks={'x': 1024, 'y': 1024}, masked = True)
            arr_resampled = clip_and_resample_raster(arr)
            arr_resampled = arr_resampled.expand_dims(time=[date])  
            resampled_arrays.append(arr_resampled)
        
        ds = xr.concat(resampled_arrays, dim='time', join = 'exact')
        yearly_mean = ds.mean(dim='time', skipna=True, keep_attrs=True)
        yearly_means.append(yearly_mean)
        del resampled_arrays, ds

    stacked_means = xr.concat(yearly_means, dim='year')

    # Calculate climatological mean
    var_climatology = stacked_means.mean(dim='year', skipna=True)
    output_path = os.path.join(out_path, "MODIS_" + variable + "_climatology.tif")
    var_climatology.rio.to_raster(
        output_path,
        dtype='float32',
        compress='lzw',
        nodata = np.nan  
    )
    
    del df, yearly_means, var_climatology

Variables:   0%|          | 0/1 [00:00<?, ?it/s]

Years (lst_diff):   0%|          | 0/25 [00:00<?, ?it/s]

In [9]:
#print list of variables prepared
print("Prepared variables for creating temperature clusters:"
      "\n1. Precipitation Climatology (CHIRPS)"
      "\n2. Fetch variables (dir and dist at 500m and 1000m)"
      "\n3. GLDAS secondary variables climatology - 'air_temp', 'lwdown', 'specific_humidity', 'surface_pressure', 'swdown', 'wind_speed'"
      "\n4. DEM variables (aspect_mean, elevation_max, elevation_mean, elevation_min, elevation_std, slope_mean)"
      "\n5. MODIS variables climatology (evi, lst_diff, lst_n, lst_d)"
      "\n All variables saved to: Data\Processed\Spatial_clustering\Climatology_files after resampling to 1km grid and reprojecting to WGS84")

Prepared variables for creating temperature clusters:
1. Precipitation Climatology (CHIRPS)
2. Fetch variables (dir and dist at 500m and 1000m)
3. GLDAS secondary variables climatology - 'air_temp', 'lwdown', 'specific_humidity', 'surface_pressure', 'swdown', 'wind_speed'
4. DEM variables (aspect_mean, elevation_max, elevation_mean, elevation_min, elevation_std, slope_mean)
5. MODIS variables climatology (evi, lst_diff, lst_n, lst_d)
 All variables saved to: Data\Processed\Spatial_clustering\Climatology_files after resampling to 1km grid and reprojecting to WGS84
